In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
from patsy import dmatrices
import seaborn as sns
import matplotlib.pyplot as plt

#### Loading data in and reformatting.

In [2]:
Metadata=pd.read_excel(r"D:\ALE06\Scripts\Viral\ALE06 dataset.xlsx", sheet_name="Visits", skiprows=2)
DORIS_Info=pd.read_excel(r"D:\ALE06\Scripts\Viral\ALE06 dataset.xlsx", sheet_name="Subjects", skiprows=1)

In [3]:
Metadata

,VisitRef,SubjectRef,Subject-Visit_ID,wd_StatusNote,Visit_ID (Schedule),Visit_ID (Sequential),Rho_USUBJID,ITTFL,ARM,RENAL,...,M6.12_Mitochondrial,M4.3_Protein Synthesis,M4.5_Protein Synthesis,M5.9_Protein Synthesis,M1.1_Platelets,M2.3_Erythrocytes,M3.1_Erythrocytes,M6.18_Erythrocytes,spacer,old_DummyAlias
0,ALE06|010021|2016-07-13,ALE06|010021,S010021-V01,NaN,W00,V01,10021.0,N,Withdrawal,Renal disease,...,0.034385,-0.033421,0.034324,0.033988,0.037532,0.034338,0.034563,-0.034327,NaN,S055-V01
1,ALE06|010021|2016-09-14,ALE06|010021,S010021-V03,NaN,W08,V03,10021.0,N,Withdrawal,Renal disease,...,0.034469,-0.034687,0.034367,0.033533,0.029870,0.028852,0.030223,-0.030186,NaN,S055-V03
2,ALE06|010021|2016-10-18,ALE06|010021,S010021-V04,NaN,W12,V04,10021.0,N,Withdrawal,Renal disease,...,0.034082,-0.034411,0.033883,0.033842,0.031229,0.028974,0.030662,-0.030033,NaN,S055-V04
3,ALE06|010021|2016-12-06,ALE06|010021,S010021-V06,NaN,W20,V06,10021.0,N,Withdrawal,Renal disease,...,0.034107,-0.034781,0.034363,0.033763,0.030486,0.028861,0.030722,-0.030094,NaN,S055-V06
4,ALE06|010021|2017-02-15,ALE06|010021,S010021-V08,NaN,W32,V08,10021.0,N,Withdrawal,Renal disease,...,0.033770,-0.033314,0.033912,0.032877,0.032182,0.028889,0.030617,-0.029833,NaN,S055-V08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
698,ALE06|080081|2017-05-23,ALE06|080081,S080081-V04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
699,ALE06|120098|2016-07-21,ALE06|120098,S120098-V02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
700,ALE06|120059|2015-06-17,ALE06|120059,S120059-V08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
701,ALE06|120098|2017-03-10,ALE06|120098,S120098-V08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Adding DORIS information to the metadata as well making a new classification

In [4]:
full_no_na=Metadata[["SubjectRef","Visit_ID (Schedule)","ARM","RENAL","STRTDOSE",
          "flaregroup","Sex","Age","EBV-VCA [Flag]","CMV [Flag]",
          "EBV EA [Flag]"]].merge(DORIS_Info[["SubjectRef","DORIS"]], how='left', on='SubjectRef').dropna()

In [5]:
flare_map={"no flare":"No_Flare","multiple flares":"Flare","oligo flare":"Flare"}
full_no_na.columns=["SubjectRef","Week","ARM","Renal","Start_Dose","flaregroup","Sex","Age","EBV_VCA","CMV","EBV_EA","DORIS"]
full_no_na["Overall_Flare"]=full_no_na["flaregroup"].map(flare_map)

In [7]:
full_no_na

,SubjectRef,Week,ARM,Renal,Start_Dose,flaregroup,Sex,Age,EBV_VCA,CMV,EBV_EA,DORIS,Overall_Flare
8,ALE06|010032,W00,Withdrawal,No renal disease,2000.0,multiple flares,M,26.0,Positive,Negative,Positive,Y,Flare
9,ALE06|010032,W12,Withdrawal,No renal disease,2000.0,multiple flares,M,26.0,Positive,Negative,Positive,Y,Flare
11,ALE06|010032,W20,Withdrawal,No renal disease,2000.0,multiple flares,M,26.0,Positive,Negative,Positive,Y,Flare
13,ALE06|010032,W32,Withdrawal,No renal disease,2000.0,multiple flares,M,26.0,Positive,Negative,Positive,Y,Flare
14,ALE06|010032,W40,Withdrawal,No renal disease,2000.0,multiple flares,M,26.0,Positive,Negative,Positive,Y,Flare
...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,ALE06|960065,END,Maintenance,Renal disease,2000.0,multiple flares,M,29.0,Negative,Negative,Negative,Y,Flare
542,ALE06|960076,W00,Withdrawal,Renal disease,2000.0,no flare,M,28.0,Positive,Negative,Positive,Y,No_Flare
543,ALE06|960076,W04,Withdrawal,Renal disease,2000.0,no flare,M,28.0,Positive,Equivocal,Positive,Y,No_Flare
544,ALE06|960076,W20,Withdrawal,Renal disease,2000.0,no flare,M,28.0,Positive,Negative,Positive,Y,No_Flare


#### Perform Logistic regression.

In [23]:
# dmatrices performs all of the one-hot encoding (Changing sex from {M,F} to {1,0}. I would probably check the matrices returned though because if you have 
# na values It'll make a unexpected column
y, X = dmatrices('Overall_Flare ~ CMV + EBV_EA + Start_Dose + Renal + Sex + Age + DORIS', 
                 data=full_no_na[((full_no_na["ARM"]=="Withdrawal")&
                                  (full_no_na["Week"]=="W00")&
                                  (full_no_na["EBV_EA"]!="Equivocal"))], return_type='dataframe')
log_reg_all = sm.Logit(np.array(y)[:,None,0], X).fit(maxiter=300000) 
log_reg_all.summary()

Optimization terminated successfully.
         Current function value: 0.521889
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:                   46
Model:                          Logit   Df Residuals:                       38
Method:                           MLE   Df Model:                            7
Date:                Mon, 17 Jun 2024   Pseudo R-squ.:                  0.2460
Time:                        15:28:28   Log-Likelihood:                -24.007
converged:                       True   LL-Null:                       -31.841
Covariance Type:            nonrobust   LLR p-value:                   0.02832
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -3.2375      2.252     -1.437      0.151      -7.652       1.177
CMV[T.Positive]            1.8180      0.916      1.986      0.047       0.023       3.613
EBV_EA[T.Positive]         1.3200      1.113      1.186      0.235      -0.861       3.501
Renal[T.Renal disease]     0.0027      0.810      0.003      0.997      -1.585       1.591
Sex[T.M]                   1.3394      1.451      0.923      0.356      -1.504       4.182
DORIS[T.Y]                -1.9362      1.040     -1.862      0.063      -3.974       0.102
Start_Dose                 0.0016      0.001      2.169      0.030       0.000       0.003
Age                       -0.0060      0.034     -0.179      0.858      -0.072       0.060
==========================================================================================
"""

In the example above "DORIS[T.Y]" means a DORIS value of Yes.

In [37]:
y, X = dmatrices('Overall_Flare ~ CMV + EBV_EA + Start_Dose + Renal + Sex + Age + DORIS', 
                 data=full_no_na[((full_no_na["ARM"]=="Maintenance")&
                                  (full_no_na["Week"]=="W00")&
                                  (full_no_na["EBV_EA"]!="Equivocal")&
                                  (full_no_na["CMV"]!="Equivocal"))], return_type='dataframe')
log_reg_all = sm.Logit(np.array(y)[:,None,0], X).fit(maxiter=300000) 
log_reg_all.summary()

Optimization terminated successfully.
         Current function value: 0.579010
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:                   41
Model:                          Logit   Df Residuals:                       33
Method:                           MLE   Df Model:                            7
Date:                Mon, 17 Jun 2024   Pseudo R-squ.:                  0.1343
Time:                        16:09:42   Log-Likelihood:                -23.739
converged:                       True   LL-Null:                       -27.423
Covariance Type:            nonrobust   LLR p-value:                    0.3917
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -0.7984      2.177     -0.367      0.714      -5.066       3.469
CMV[T.Positive]           -0.6239      0.902     -0.691      0.489      -2.392       1.145
EBV_EA[T.Positive]         0.0976      0.959      0.102      0.919      -1.783       1.978
Renal[T.Renal disease]    -0.0502      0.878     -0.057      0.954      -1.771       1.671
Sex[T.M]                  -1.6716      1.237     -1.351      0.177      -4.096       0.753
DORIS[T.Y]                -1.2150      0.946     -1.284      0.199      -3.070       0.640
Start_Dose                -0.0002      0.001     -0.253      0.801      -0.001       0.001
Age                        0.0494      0.036      1.389      0.165      -0.020       0.119
==========================================================================================
"""

Cannot include EBV-VCA because it results in a singular matrix

In [25]:
pd.crosstab(full_no_na[((full_no_na["ARM"]=="Withdrawal")&
                                  (full_no_na["Week"]=="W00"))]["EBV_VCA"],
                                  full_no_na[((full_no_na["ARM"]=="Withdrawal")&
                                  (full_no_na["Week"]=="W00"))]["Overall_Flare"],margins=True)

Overall_Flare,Flare,No_Flare,All
EBV_VCA,,,
Negative,0,1,1
Positive,24,23,47
All,24,24,48


In [35]:
pd.crosstab(full_no_na[((full_no_na["ARM"]=="Withdrawal")&
                                  (full_no_na["Week"]=="W00"))]["DORIS"],
                                  full_no_na[((full_no_na["ARM"]=="Withdrawal")&
                                  (full_no_na["Week"]=="W00"))]["Overall_Flare"],margins=True)

Overall_Flare,Flare,No_Flare,All
DORIS,,,
N,8,3,11
Y,16,21,37
All,24,24,48
